In [1]:
#!pip install kagglehub datasets -q

import os, re, warnings
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from datasets import load_dataset
warnings.filterwarnings("ignore")
print("Ready.")

Ready.


In [2]:
# hendrycks/ethics is gated forever — use MMLU moral splits instead
# All 8 URLs below are confirmed HTTP 200 public parquets

MMLU_MORAL_URLS = {
    "moral_scenarios_test":  "https://huggingface.co/datasets/cais/mmlu/resolve/main/moral_scenarios/test-00000-of-00001.parquet",
    "moral_scenarios_val":   "https://huggingface.co/datasets/cais/mmlu/resolve/main/moral_scenarios/validation-00000-of-00001.parquet",
    "moral_disputes_test":   "https://huggingface.co/datasets/cais/mmlu/resolve/main/moral_disputes/test-00000-of-00001.parquet",
    "moral_disputes_val":    "https://huggingface.co/datasets/cais/mmlu/resolve/main/moral_disputes/validation-00000-of-00001.parquet",
    "professional_law":      "https://huggingface.co/datasets/cais/mmlu/resolve/main/professional_law/test-00000-of-00001.parquet",
    "philosophy":            "https://huggingface.co/datasets/cais/mmlu/resolve/main/philosophy/test-00000-of-00001.parquet",
    "sociology":             "https://huggingface.co/datasets/cais/mmlu/resolve/main/sociology/test-00000-of-00001.parquet",
    "public_relations":      "https://huggingface.co/datasets/cais/mmlu/resolve/main/public_relations/test-00000-of-00001.parquet",
}

ethics_rows = []
for name, url in MMLU_MORAL_URLS.items():
    try:
        df = pd.read_parquet(url)
        df.columns = [c.strip().lower() for c in df.columns]
        # MMLU answer column: 0=A,1=B,2=C,3=D — "correct" answer exists as answer col
        answer_col = next((c for c in df.columns if "answer" in c), None)
        if answer_col:
            # Distribution of correct answers as alignment proxy
            rate = float((pd.to_numeric(df[answer_col], errors="coerce") < 2).mean())
        else:
            rate = 0.5
        ethics_rows.append({"split": name, "moral_acceptance_rate": rate, "n": len(df)})
        print(f"[OK] {name:30s} n={len(df):4d}  cols={list(df.columns)}")
    except Exception as e:
        print(f"[WARN] {name}: {e}")

df_ethics_summary = pd.DataFrame(ethics_rows)
moral_signal = float(df_ethics_summary["moral_acceptance_rate"].mean()) \
               if not df_ethics_summary.empty else 0.5
print(f"\nmoral_signal = {moral_signal:.4f}")
print(df_ethics_summary.to_string(index=False))

[OK] moral_scenarios_test           n= 895  cols=['question', 'subject', 'choices', 'answer']
[OK] moral_scenarios_val            n= 100  cols=['question', 'subject', 'choices', 'answer']
[OK] moral_disputes_test            n= 346  cols=['question', 'subject', 'choices', 'answer']
[OK] moral_disputes_val             n=  38  cols=['question', 'subject', 'choices', 'answer']
[OK] professional_law               n=1534  cols=['question', 'subject', 'choices', 'answer']
[OK] philosophy                     n= 311  cols=['question', 'subject', 'choices', 'answer']
[OK] sociology                      n= 201  cols=['question', 'subject', 'choices', 'answer']
[OK] public_relations               n= 110  cols=['question', 'subject', 'choices', 'answer']

moral_signal = 0.5053
               split  moral_acceptance_rate    n
moral_scenarios_test               0.480447  895
 moral_scenarios_val               0.550000  100
 moral_disputes_test               0.494220  346
  moral_disputes_val         

In [3]:
ds_dolly = None
df_dolly = pd.DataFrame()
try:
    ds_dolly = load_dataset("mosaicml/dolly_hhrlhf")
    split = "train" if "train" in ds_dolly else list(ds_dolly.keys())[0]
    df_dolly = pd.DataFrame(ds_dolly[split]).head(10_000)
    print(f"[OK] dolly_hhrlhf  shape={df_dolly.shape}  cols={list(df_dolly.columns)}")
except Exception as e:
    print(f"[WARN] dolly_hhrlhf: {e}")

[OK] dolly_hhrlhf  shape=(10000, 2)  cols=['prompt', 'response']


In [4]:
ds_finclass = None
df_finclass = pd.DataFrame()
try:
    ds_finclass = load_dataset("nickmuchi/financial-classification")
    split = "train" if "train" in ds_finclass else list(ds_finclass.keys())[0]
    df_finclass = pd.DataFrame(ds_finclass[split]).head(10_000)
    df_finclass.columns = [c.strip().lower() for c in df_finclass.columns]
    print(f"[OK] financial-classification  shape={df_finclass.shape}  cols={list(df_finclass.columns)}")
    label_col = next((c for c in df_finclass.columns if "label" in c), None)
    if label_col:
        print(f"     label dist:\n{df_finclass[label_col].value_counts().to_string()}")
except Exception as e:
    print(f"[WARN] financial-classification: {e}")

[OK] financial-classification  shape=(4551, 2)  cols=['text', 'labels']
     label dist:
labels
1    2676
2    1274
0     601


In [5]:
df_covid = pd.DataFrame()
covid_disruption_signal = 0.05

try:
    mount_root = kagglehub.dataset_download(
        "sudalairajkumar/novel-corona-virus-2019-dataset")
    confirmed_path = os.path.join(mount_root, "time_series_covid_19_confirmed.csv")
    df_raw = pd.read_csv(confirmed_path, low_memory=False)
    df_raw.columns = [c.strip() for c in df_raw.columns]

    # Wide format: date columns look like 1/22/20
    date_cols = [c for c in df_raw.columns if re.match(r"\d+/\d+/\d+", str(c))]
    print(f"[OK] covid  shape={df_raw.shape}  date_cols={len(date_cols)}")

    global_ts = df_raw[date_cols].sum(axis=0)
    global_ts.index = pd.to_datetime(global_ts.index, format="%m/%d/%y", errors="coerce")
    global_ts = global_ts.sort_index()

    daily_growth = global_ts.pct_change().clip(-1, 1)
    covid_disruption_signal = float(daily_growth.abs().rolling(7).mean().max())
    print(f"     peak 7-day disruption signal: {covid_disruption_signal:.4f}")
    print(f"     peak date: {daily_growth.abs().rolling(7).mean().idxmax().date()}")
except Exception as e:
    print(f"[WARN] covid: {e}")

print(f"\ncovid_disruption_signal = {covid_disruption_signal:.4f}")

[OK] covid  shape=(276, 498)  date_cols=494
     peak 7-day disruption signal: 0.4523
     peak date: 2020-01-30

covid_disruption_signal = 0.4523


In [6]:
df_uspto = pd.DataFrame()
ip_infrastructure_signal = 0.03

try:
    mount_root = kagglehub.dataset_download(
        "uspto/us-trademark-case-files-18702016")

    # Read only the filing_dt column from case_file.csv — avoids OOM
    case_path = os.path.join(mount_root, "case_file.csv")
    header_df = pd.read_csv(case_path, nrows=0, dtype=str)
    all_cols  = list(header_df.columns)
    date_col  = next((c for c in all_cols
                      if any(k in c.lower() for k in ["filing","regist","date"])), None)
    print(f"[OK] case_file.csv  cols={len(all_cols)}  using: {date_col}")

    df_filing = pd.read_csv(case_path, usecols=[date_col], dtype=str, low_memory=False)
    yr = pd.to_numeric(
        pd.to_datetime(df_filing[date_col], errors="coerce").dt.year,
        errors="coerce").dropna()
    yr = yr[(yr >= 1870) & (yr <= 2020)].astype(int)

    annual = yr.value_counts().sort_index()
    roll   = annual.pct_change().clip(-1,1).rolling(5).mean().dropna()
    ip_infrastructure_signal = float(roll.iloc[-1]) if len(roll) > 0 else 0.03
    print(f"     total filings: {len(yr):,}  peak year: {annual.idxmax()} ({annual.max():,})")
    print(f"     5yr avg growth: {ip_infrastructure_signal:.4f}")
except Exception as e:
    print(f"[WARN] uspto: {e}")

print(f"\nip_infrastructure_signal = {ip_infrastructure_signal:.4f}")

[OK] case_file.csv  cols=79  using: filing_dt
     total filings: 7,621,528  peak year: 2016 (390,829)
     5yr avg growth: -0.1346

ip_infrastructure_signal = -0.1346


In [15]:
# SDGData.csv mixes indicators with wildly different scales (GDP, %, ratios, counts)
# Fix: filter to only 0-100 range indicators (percentages, index scores, rates)
# which are the meaningful human-progress signals

import numpy as np
import pandas as pd
import os, re

sdg_root = "/kaggle/input/datasets/organizations/theworldbank/sustainable-development-goals"
sdg_path = os.path.join(sdg_root, "sdg-csv-zip-7-mb-", "SDGData.csv")

df_sdg = pd.read_csv(sdg_path, low_memory=False, dtype=str, nrows=300_000)
df_sdg.columns = [re.sub(r"[^a-z0-9]+","_",c.strip().lower()).strip("_")
                  for c in df_sdg.columns]

yr_cols      = [c for c in df_sdg.columns if re.match(r"^\d{4}$", c)]
recent_yrs   = [c for c in yr_cols if int(c) >= 2015]
id_cols      = [c for c in df_sdg.columns if c not in yr_cols]

print(f"SDGData shape={df_sdg.shape}  recent_yrs={recent_yrs}")
print(f"id cols: {id_cols}")

# Convert year cols to numeric
for c in recent_yrs:
    df_sdg[c] = pd.to_numeric(df_sdg[c], errors="coerce")

# Compute per-indicator median across recent years
df_sdg["recent_mean"] = df_sdg[recent_yrs].median(axis=1)

# Show the scale distribution
print("\nRecent mean distribution:")
print(df_sdg["recent_mean"].describe())

# Identify indicator name column
ind_col = next((c for c in id_cols if "indicator" in c and "code" not in c), None)
print(f"\nIndicator col: {ind_col}")
if ind_col:
    print("\nTop 10 largest-value indicators (to understand what's skewing the mean):")
    print(df_sdg.nlargest(10, "recent_mean")[[ind_col, "recent_mean"]].to_string(index=False))

# Strategy: keep only indicators whose recent_mean is in [0, 100]
# These are rates, percentages, index scores — the meaningful human-progress signals
df_pct = df_sdg[
    (df_sdg["recent_mean"] >= 0) &
    (df_sdg["recent_mean"] <= 100) &
    (df_sdg["recent_mean"].notna())
].copy()

print(f"\nIndicators in 0-100 range: {len(df_pct)} / {len(df_sdg)}")

sdg_signal = float(df_pct["recent_mean"].mean()) if len(df_pct) > 0 else 0.0
print(f"sdg_signal (normalized 0-100 indicators only) = {sdg_signal:.4f}")

# Also show what kinds of indicators made the cut
if ind_col and len(df_pct) > 0:
    print("\nSample indicators included:")
    for s in df_pct[ind_col].dropna().sample(min(8, len(df_pct)), random_state=42).tolist():
        print(f"  {s[:90]}")

SDGData shape=(98625, 34)  recent_yrs=['2015', '2016', '2017', '2018']
id cols: ['country_name', 'country_code', 'indicator_name', 'indicator_code', 'unnamed_33']

Recent mean distribution:
count    5.370500e+04
mean     1.699395e+12
std      1.096801e+14
min     -2.426612e+10
25%      6.044670e+00
50%      2.954671e+01
75%      9.825902e+01
max      1.315126e+16
Name: recent_mean, dtype: float64

Indicator col: indicator_name

Top 10 largest-value indicators (to understand what's skewing the mean):
    indicator_name  recent_mean
 GDP (current LCU) 1.315126e+16
 GDP (current LCU) 1.240173e+16
GDP (constant LCU) 9.434613e+15
GNI (constant LCU) 9.144229e+15
GNI (constant LCU) 6.933341e+15
GDP (constant LCU) 6.916081e+15
 GDP (current LCU) 4.502733e+15
GDP (constant LCU) 3.054470e+15
GNI (constant LCU) 2.920314e+15
 GDP (current LCU) 1.641786e+15

Indicators in 0-100 range: 42247 / 98625
sdg_signal (normalized 0-100 indicators only) = 31.4000

Sample indicators included:
  Unemployment, 

In [18]:
# All signals must be on the same scale for the weighted sum in inject_signals()
# Current scales:
#   moral_alignment:   0.5053  ← already 0-1 ✅
#   covid_disruption:  0.4523  ← already 0-1 ✅
#   ip_infrastructure: -0.1346 ← already ~0-1 range ✅
#   global_infra:      0.7433  ← already 0-1 ✅
#   sdg_progress:      31.4    ← 0-100 scale ❌ needs /100

sdg_signal_raw = sdg_signal
sdg_signal     = sdg_signal / 100.0   # normalize to 0-1

# hdi, happiness, market_sentiment are already 0-1 when they come in from earlier cells
# ip_infrastructure is a growth rate so leave it as-is (small negative is fine)

print("── Normalized signal inventory ───────────────────────────")
for k, v in [
    ("moral_alignment",    moral_signal),
    ("covid_disruption",   covid_disruption_signal),
    ("ip_infrastructure",  ip_infrastructure_signal),
    ("global_infra_index", infra_signal),
    ("sdg_progress",       sdg_signal),
    ("hdi_growth",         hdi_scalar),
    ("happiness_growth",   happiness_scalar),
    ("market_sentiment",   market_sentiment_signal),
]:
    bar = "█" * int(abs(v) * 20)
    sign = "-" if v < 0 else " "
    print(f"  {k:30s}  {sign}{bar:<20s}  {v:+.4f}")

print(f"\n  sdg_signal: {sdg_signal_raw:.2f} (raw) → {sdg_signal:.4f} (normalized)")
print("\n✅ All signals ready — run the Granger scoring cell, then call:")
print("   causal_df = inject_signals(causal_df)")

── Normalized signal inventory ───────────────────────────
  moral_alignment                  ██████████            +0.5053
  covid_disruption                 █████████             +0.4523
  ip_infrastructure               -██                    -0.1346
  global_infra_index               ██████████████        +0.7433
  sdg_progress                     ██████                +0.3140
  hdi_growth                                             +0.0000
  happiness_growth                                       +0.0000
  market_sentiment                                       +0.0000

  sdg_signal: 31.40 (raw) → 0.3140 (normalized)

✅ All signals ready — run the Granger scoring cell, then call:
   causal_df = inject_signals(causal_df)


In [19]:
SECTOR_DISRUPTION = {
    "Health Care":1.0,"Consumer Staples":0.9,"Utilities":0.8,"Real Estate":0.7,
    "Financials":0.6,"Industrials":0.7,"Consumer Discretionary":0.8,
    "Information Technology":0.5,"Energy":0.6,"Materials":0.6,
    "Communication Services":0.4,
}
SECTOR_IP = {
    "Information Technology":1.0,"Health Care":0.9,"Communication Services":0.8,
    "Industrials":0.6,"Consumer Discretionary":0.5,"Materials":0.4,
    "Financials":0.3,"Utilities":0.2,"Real Estate":0.2,
    "Energy":0.4,"Consumer Staples":0.3,
}
SECTOR_GLOBAL = {
    "Information Technology":1.0,"Financials":0.9,"Health Care":0.7,
    "Consumer Discretionary":0.6,"Industrials":0.7,"Materials":0.8,
    "Energy":0.8,"Consumer Staples":0.5,"Communication Services":0.7,
    "Utilities":0.3,"Real Estate":0.2,
}

def inject_signals(df):
    df = df.copy()
    s = df["sector"]
    df["moral_alignment_signal"]   = moral_signal
    df["covid_disruption_signal"]  = s.map(lambda x: SECTOR_DISRUPTION.get(x,0.5)*covid_disruption_signal)
    df["ip_infrastructure_signal"] = s.map(lambda x: SECTOR_IP.get(x,0.5)*ip_infrastructure_signal)
    df["global_infra_signal"]      = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)*infra_signal)
    df["sdg_progress_signal"]      = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)*sdg_signal)
    df["hdi_growth_signal"]        = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)*hdi_scalar)
    df["happiness_growth_signal"]  = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)*happiness_scalar)
    df["market_sentiment_signal"]  = market_sentiment_signal

    df["human_progress_enrichment"] = (
        0.18 * df["hdi_growth_signal"].fillna(0) +
        0.12 * df["happiness_growth_signal"].fillna(0) +
        0.15 * df["moral_alignment_signal"].fillna(0) +
        0.15 * df["ip_infrastructure_signal"].fillna(0) +
        0.15 * df["global_infra_signal"].fillna(0) +
        0.12 * df["sdg_progress_signal"].fillna(0) +
        0.08 * df["market_sentiment_signal"].fillna(0) -
        0.05 * df["covid_disruption_signal"].fillna(0)
    )
    enrich_z = df["human_progress_enrichment"]
    enrich_z = (enrich_z - enrich_z.mean()) / (enrich_z.std() + 1e-9)
    df["causal_score_enriched"] = (
        0.70 * df["causal_score"].fillna(0) +
        0.30 * enrich_z.fillna(0)
    )
    return df.sort_values("causal_score_enriched", ascending=False).reset_index(drop=True)

# Apply immediately if causal_df already exists
if "causal_df" in globals() and not causal_df.empty:
    causal_df = inject_signals(causal_df)
    print(f"[OK] causal_df enriched  shape={causal_df.shape}")
    display(causal_df[["ticker","sector","causal_score",
                        "causal_score_enriched","human_progress_enrichment"]].head(15))
else:
    print("[INFO] inject_signals() ready — call causal_df = inject_signals(causal_df) after Granger cell")

[INFO] inject_signals() ready — call causal_df = inject_signals(causal_df) after Granger cell


In [30]:
import os, re
import pandas as pd
import numpy as np

# ── Fix 1: happiness_scalar was 7.8 (raw 0-10 score) — normalize to 0-1
happiness_scalar_raw = globals().get("happiness_scalar", 7.8)
happiness_scalar = happiness_scalar_raw / 10.0 if happiness_scalar_raw > 1.0 else happiness_scalar_raw
print(f"happiness_scalar: {happiness_scalar_raw} → {happiness_scalar:.4f}")

# ── Fix 2: hdi_scalar was 0.9431 which is mean IHDI rank/195
# Load the correct file: human_development.csv has actual HDI values (0-1 scale)
hdi_root = "/kaggle/input/datasets/organizations/undp/human-development"
hdi_path = os.path.join(hdi_root, "human_development.csv")
hdi_scalar = 0.0

try:
    df_hdi = pd.read_csv(hdi_path, low_memory=False, dtype=str)
    df_hdi.columns = [re.sub(r"[^a-z0-9]+","_",c.strip().lower()).strip("_")
                      for c in df_hdi.columns]
    print(f"\n[OK] human_development.csv  shape={df_hdi.shape}")
    print(f"     cols: {list(df_hdi.columns)}")

    # Find the HDI value column (should be 0-1 range)
    val_col = next((c for c in df_hdi.columns
                    if "human_development_index" in c or c == "hdi_value"
                    or (c.startswith("hdi") and "rank" not in c)), None)
    print(f"     val_col: {val_col}")

    if val_col:
        vals = pd.to_numeric(df_hdi[val_col], errors="coerce").dropna()
        # HDI values are 0-1 — filter out anything > 1 (ranks/years accidentally matched)
        hdi_vals = vals[vals <= 1.0]
        if len(hdi_vals) > 0:
            hdi_scalar = float(hdi_vals.mean())
            print(f"     HDI values (0-1): n={len(hdi_vals)}  mean={hdi_scalar:.4f}")
            print(f"     min={hdi_vals.min():.3f}  max={hdi_vals.max():.3f}")
        else:
            # All values > 1 — likely stored as e.g. 0.800 but read as 800
            hdi_scalar = float(vals.mean()) / 1000.0
            print(f"     Scaled fallback: {hdi_scalar:.4f}")
    else:
        print(f"     [WARN] no HDI value col found — trying all numeric cols:")
        num_cols = df_hdi.apply(pd.to_numeric, errors="coerce")
        for c in num_cols.columns:
            v = num_cols[c].dropna()
            if len(v) > 50 and v.between(0,1).mean() > 0.8:
                hdi_scalar = float(v.mean())
                print(f"     Using col '{c}'  mean={hdi_scalar:.4f}")
                break
except Exception as e:
    print(f"[WARN] HDI fix: {e}")
    # Sensible global average HDI (~0.73 as of 2022 UNDP report)
    hdi_scalar = 0.73
    print(f"     Using known global average: {hdi_scalar:.4f}")

print(f"\n── Corrected scalars ──────────────────────")
print(f"  hdi_scalar       = {hdi_scalar:.4f}  (was {globals().get('hdi_scalar',0):.4f})")
print(f"  happiness_scalar = {happiness_scalar:.4f}  (was {happiness_scalar_raw:.4f})")

# ── Re-inject with corrected values
# (inject_signals() already defined in Cell M — just re-run it)
causal_df = inject_signals(causal_df.drop(
    columns=[c for c in causal_df.columns
             if "signal" in c or "enrich" in c], errors="ignore"))

print(f"\n[OK] causal_df re-enriched  shape={causal_df.shape}")
display(causal_df[[
    "ticker","sector","causal_score",
    "causal_score_enriched","human_progress_enrichment"
]].head(10))

# ── Re-save everything
os.makedirs("/kaggle/working", exist_ok=True)
causal_df.to_csv("/kaggle/working/causal_df_enriched.csv", index=False)

top50 = causal_df.head(50).copy()
top50.insert(0, "rank", range(1, len(top50)+1))
top50.to_csv("/kaggle/working/top50_causal_leaderboard.csv", index=False)

signal_manifest = pd.DataFrame([
    ("moral_alignment",    moral_signal,            "MMLU moral splits — 8 confirmed parquets"),
    ("covid_disruption",   covid_disruption_signal, "Sudalairajkumar COVID-19 confirmed time series"),
    ("ip_infrastructure",  ip_infrastructure_signal,"USPTO trademark case files 1870-2016"),
    ("global_infra_index", infra_signal,            "Rodrigogaluppo Global Infrastructure Index"),
    ("sdg_progress",       sdg_signal,              "World Bank SDGData.csv (0-100 indicators normalized)"),
    ("hdi_growth",         hdi_scalar,              "UNDP human_development.csv — global mean HDI (0-1)"),
    ("happiness_level",    happiness_scalar,        "UN SDSN World Happiness 2017 — normalized 0-1"),
    ("market_sentiment",   market_sentiment_signal, "Nickmuchi financial-classification"),
], columns=["signal","value","source"])
signal_manifest.to_csv("/kaggle/working/signal_manifest.csv", index=False)

print("\n✅ Re-saved corrected outputs:")
print("   causal_df_enriched.csv")
print("   top50_causal_leaderboard.csv")
print("   signal_manifest.csv")

happiness_scalar: 7.8 → 0.7800

[OK] human_development.csv  shape=(195, 8)
     cols: ['hdi_rank', 'country', 'human_development_index_hdi', 'life_expectancy_at_birth', 'expected_years_of_education', 'mean_years_of_education', 'gross_national_income_gni_per_capita', 'gni_per_capita_rank_minus_hdi_rank']
     val_col: human_development_index_hdi
     HDI values (0-1): n=195  mean=0.6918
     min=0.348  max=0.944

── Corrected scalars ──────────────────────
  hdi_scalar       = 0.6918  (was 0.6918)
  happiness_scalar = 0.7800  (was 7.8000)

[OK] causal_df re-enriched  shape=(40, 15)


,ticker,sector,causal_score,causal_score_enriched,human_progress_enrichment
0,AAPL,Information Technology,0.878303,0.995901,0.382666
1,MSFT,Information Technology,0.814466,0.951215,0.382666
2,JPM,Financials,0.939686,0.941996,0.359933
3,BAC,Financials,0.883147,0.902419,0.359933
4,ORCL,Information Technology,0.698664,0.870153,0.382666
5,INTC,Information Technology,0.670002,0.850090,0.382666
6,GS,Financials,0.760017,0.816228,0.359933
7,APD,Materials,0.936784,0.791166,0.325014
8,CVX,Energy,0.882694,0.753303,0.325014
9,EOG,Energy,0.847210,0.728464,0.325014



✅ Re-saved corrected outputs:
   causal_df_enriched.csv
   top50_causal_leaderboard.csv
   signal_manifest.csv


In [31]:
import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings("ignore")

# ── Ticker universe with sectors
TICKERS = {
    # Information Technology
    "AAPL":"Information Technology","MSFT":"Information Technology",
    "NVDA":"Information Technology","GOOGL":"Communication Services",
    "META":"Communication Services","AMZN":"Consumer Discretionary",
    "TSLA":"Consumer Discretionary","JPM":"Financials",
    "JNJ":"Health Care","UNH":"Health Care",
    "XOM":"Energy","CVX":"Energy",
    "CAT":"Industrials","HON":"Industrials",
    "PG":"Consumer Staples","KO":"Consumer Staples",
    "NEE":"Utilities","DUK":"Utilities",
    "AMT":"Real Estate","PLD":"Real Estate",
    "LIN":"Materials","APD":"Materials",
    "NFLX":"Communication Services","DIS":"Communication Services",
    "BAC":"Financials","GS":"Financials",
    "PFE":"Health Care","ABBV":"Health Care",
    "INTC":"Information Technology","AMD":"Information Technology",
    "CRM":"Information Technology","ORCL":"Information Technology",
    "WMT":"Consumer Staples","COST":"Consumer Staples",
    "BA":"Industrials","GE":"Industrials",
    "SLB":"Energy","EOG":"Energy",
    "FCX":"Materials","NEM":"Materials",
}

print(f"Downloading price data for {len(TICKERS)} tickers...")

# ── Download prices
try:
    raw = yf.download(
        list(TICKERS.keys()),
        start="2018-01-01", end="2024-12-31",
        auto_adjust=True, progress=False
    )
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw["Close"].dropna(how="all")
    else:
        prices = raw[["Close"]].dropna()
    print(f"[OK] prices  shape={prices.shape}")
except Exception as e:
    print(f"[WARN] yfinance: {e}")
    prices = pd.DataFrame()

if prices.empty:
    print("[WARN] No price data — creating synthetic causal_df for testing")
    rows = []
    for ticker, sector in TICKERS.items():
        rows.append({
            "ticker": ticker, "sector": sector,
            "causal_score": np.random.uniform(0.1, 0.9),
            "granger_pval": np.random.uniform(0.01, 0.5),
            "n_caused": np.random.randint(1, 10),
        })
    causal_df = pd.DataFrame(rows).sort_values("causal_score", ascending=False).reset_index(drop=True)
    print(f"[OK] synthetic causal_df  shape={causal_df.shape}")
else:
    # ── Compute returns
    returns = prices.pct_change().dropna()
    tickers_avail = [t for t in TICKERS if t in returns.columns]
    print(f"     tickers available: {len(tickers_avail)}")

    # ── Granger causality: does ticker A Granger-cause ticker B?
    MAXLAG = 5
    results = []
    total_pairs = len(tickers_avail) ** 2
    done = 0

    for cause in tickers_avail:
        cause_score = 0
        n_caused    = 0
        pvals       = []
        for effect in tickers_avail:
            if cause == effect: continue
            try:
                data = pd.concat([returns[effect], returns[cause]], axis=1).dropna()
                if len(data) < 50: continue
                gc = grangercausalitytests(data.values, maxlag=MAXLAG, verbose=False)
                # Take minimum p-value across lags
                min_pval = min(
                    gc[lag][0]["ssr_ftest"][1] for lag in range(1, MAXLAG+1))
                if min_pval < 0.05:
                    cause_score += (1 - min_pval)
                    n_caused    += 1
                pvals.append(min_pval)
            except Exception:
                continue
        done += 1
        if done % 5 == 0:
            print(f"     {done}/{len(tickers_avail)} tickers processed...")

        mean_pval = float(np.mean(pvals)) if pvals else 1.0
        results.append({
            "ticker":       cause,
            "sector":       TICKERS.get(cause, "Unknown"),
            "causal_score": round(cause_score, 4),
            "granger_pval": round(mean_pval, 4),
            "n_caused":     n_caused,
        })

    causal_df = pd.DataFrame(results)
    # Normalize causal_score to 0-1
    cs_min = causal_df["causal_score"].min()
    cs_max = causal_df["causal_score"].max()
    causal_df["causal_score"] = ((causal_df["causal_score"] - cs_min) /
                                  (cs_max - cs_min + 1e-9)).round(6)
    causal_df = causal_df.sort_values("causal_score", ascending=False).reset_index(drop=True)
    print(f"\n[OK] causal_df  shape={causal_df.shape}")

print("\nTop 15 by raw causal_score:")
display(causal_df[["ticker","sector","causal_score","granger_pval","n_caused"]].head(15))

[OK] prices  shape=(1760, 40)
     tickers available: 40
     5/40 tickers processed...
     10/40 tickers processed...
     15/40 tickers processed...
     20/40 tickers processed...
     25/40 tickers processed...
     30/40 tickers processed...
     35/40 tickers processed...
     40/40 tickers processed...

[OK] causal_df  shape=(40, 5)

Top 15 by raw causal_score:


,ticker,sector,causal_score,granger_pval,n_caused
0,UNH,Health Care,1.000000,0.0021,39
1,DUK,Utilities,0.996792,0.0048,39
2,JPM,Financials,0.939686,0.0117,37
3,APD,Materials,0.936784,0.0108,37
4,PG,Consumer Staples,0.912003,0.0115,36
5,BAC,Financials,0.883147,0.0245,35
6,CVX,Energy,0.882694,0.0170,35
7,AAPL,Information Technology,0.878303,0.0385,35
8,NEE,Utilities,0.877559,0.0272,35
9,EOG,Energy,0.847210,0.0338,34


In [32]:
# Re-sync all signals from globals in case any cell was re-run
def _g(k, d=0.0): return globals().get(k, d)

moral_signal             = _g("moral_signal",             0.5)
covid_disruption_signal  = _g("covid_disruption_signal",  0.05)
ip_infrastructure_signal = _g("ip_infrastructure_signal", 0.03)
infra_signal             = _g("infra_signal",             0.0)
sdg_signal               = _g("sdg_signal",               0.0)
market_sentiment_signal  = _g("market_sentiment_signal",  0.0)
hdi_scalar               = _g("hdi_scalar",               0.0)
happiness_scalar         = _g("happiness_scalar",         0.0)

SECTOR_DISRUPTION = {
    "Health Care":1.0,"Consumer Staples":0.9,"Utilities":0.8,"Real Estate":0.7,
    "Financials":0.6,"Industrials":0.7,"Consumer Discretionary":0.8,
    "Information Technology":0.5,"Energy":0.6,"Materials":0.6,
    "Communication Services":0.4,
}
SECTOR_IP = {
    "Information Technology":1.0,"Health Care":0.9,"Communication Services":0.8,
    "Industrials":0.6,"Consumer Discretionary":0.5,"Materials":0.4,
    "Financials":0.3,"Utilities":0.2,"Real Estate":0.2,
    "Energy":0.4,"Consumer Staples":0.3,
}
SECTOR_GLOBAL = {
    "Information Technology":1.0,"Financials":0.9,"Health Care":0.7,
    "Consumer Discretionary":0.6,"Industrials":0.7,"Materials":0.8,
    "Energy":0.8,"Consumer Staples":0.5,"Communication Services":0.7,
    "Utilities":0.3,"Real Estate":0.2,
}

def inject_signals(df):
    df = df.copy()
    s = df["sector"]
    df["moral_alignment_signal"]   = moral_signal
    df["covid_disruption_signal"]  = s.map(lambda x: SECTOR_DISRUPTION.get(x,0.5) * covid_disruption_signal)
    df["ip_infrastructure_signal"] = s.map(lambda x: SECTOR_IP.get(x,0.5)         * ip_infrastructure_signal)
    df["global_infra_signal"]      = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)     * infra_signal)
    df["sdg_progress_signal"]      = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)     * sdg_signal)
    df["hdi_growth_signal"]        = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)     * hdi_scalar)
    df["happiness_signal"]         = s.map(lambda x: SECTOR_GLOBAL.get(x,0.5)     * happiness_scalar)
    df["market_sentiment_signal"]  = market_sentiment_signal

    df["human_progress_enrichment"] = (
        0.15 * df["hdi_growth_signal"].fillna(0) +
        0.12 * df["happiness_signal"].fillna(0) +
        0.13 * df["moral_alignment_signal"].fillna(0) +
        0.13 * df["ip_infrastructure_signal"].fillna(0) +
        0.13 * df["global_infra_signal"].fillna(0) +
        0.12 * df["sdg_progress_signal"].fillna(0) +
        0.08 * df["market_sentiment_signal"].fillna(0) -
        0.04 * df["covid_disruption_signal"].fillna(0)
    )

    enrich_z = df["human_progress_enrichment"]
    enrich_z = (enrich_z - enrich_z.mean()) / (enrich_z.std() + 1e-9)

    df["causal_score_enriched"] = (
        0.70 * df["causal_score"].fillna(0) +
        0.30 * enrich_z.fillna(0)
    ).round(6)

    return df.sort_values("causal_score_enriched", ascending=False).reset_index(drop=True)

causal_df = inject_signals(causal_df)
print(f"[OK] causal_df enriched  shape={causal_df.shape}")
print(f"     signal cols: {[c for c in causal_df.columns if 'signal' in c or 'enrich' in c]}")
display(causal_df[[
    "ticker","sector","causal_score",
    "causal_score_enriched","human_progress_enrichment"
]].head(20))

[OK] causal_df enriched  shape=(40, 15)
     signal cols: ['moral_alignment_signal', 'covid_disruption_signal', 'ip_infrastructure_signal', 'global_infra_signal', 'sdg_progress_signal', 'hdi_growth_signal', 'happiness_signal', 'market_sentiment_signal', 'human_progress_enrichment', 'causal_score_enriched']


,ticker,sector,causal_score,causal_score_enriched,human_progress_enrichment
0,AAPL,Information Technology,0.878303,0.995901,0.382666
1,MSFT,Information Technology,0.814466,0.951215,0.382666
2,JPM,Financials,0.939686,0.941996,0.359933
3,BAC,Financials,0.883147,0.902419,0.359933
4,ORCL,Information Technology,0.698664,0.870153,0.382666
5,INTC,Information Technology,0.670002,0.850090,0.382666
6,GS,Financials,0.760017,0.816228,0.359933
7,APD,Materials,0.936784,0.791166,0.325014
8,CVX,Energy,0.882694,0.753303,0.325014
9,EOG,Energy,0.847210,0.728464,0.325014


In [33]:
import os
os.makedirs("/kaggle/working", exist_ok=True)

causal_df.to_csv("/kaggle/working/causal_df_enriched.csv", index=False)

top50 = causal_df.head(50).copy()
top50.insert(0, "rank", range(1, len(top50)+1))
top50.to_csv("/kaggle/working/top50_causal_leaderboard.csv", index=False)

signal_manifest = pd.DataFrame([
    ("moral_alignment",    moral_signal,            "MMLU moral splits — 8 confirmed parquets"),
    ("covid_disruption",   covid_disruption_signal, "Sudalairajkumar COVID-19 confirmed time series"),
    ("ip_infrastructure",  ip_infrastructure_signal,"USPTO trademark case files 1870-2016"),
    ("global_infra_index", infra_signal,            "Rodrigogaluppo Global Infrastructure Index"),
    ("sdg_progress",       sdg_signal,              "World Bank SDGData.csv (0-100 indicators normalized)"),
    ("hdi_growth",         hdi_scalar,              "UNDP Human Development Index"),
    ("happiness_level",    happiness_scalar,        "UN SDSN World Happiness Report"),
    ("market_sentiment",   market_sentiment_signal, "Nickmuchi financial-classification"),
], columns=["signal","value","source"])
signal_manifest.to_csv("/kaggle/working/signal_manifest.csv", index=False)

print("✅ Saved to /kaggle/working/:")
print("   causal_df_enriched.csv")
print("   top50_causal_leaderboard.csv")
print("   signal_manifest.csv")
print(f"\nFinal top 5:")
display(top50[["rank","ticker","sector","causal_score_enriched","human_progress_enrichment"]].head(5))

✅ Saved to /kaggle/working/:
   causal_df_enriched.csv
   top50_causal_leaderboard.csv
   signal_manifest.csv

Final top 5:


,rank,ticker,sector,causal_score_enriched,human_progress_enrichment
0,1,AAPL,Information Technology,0.995901,0.382666
1,2,MSFT,Information Technology,0.951215,0.382666
2,3,JPM,Financials,0.941996,0.359933
3,4,BAC,Financials,0.902419,0.359933
4,5,ORCL,Information Technology,0.870153,0.382666
